# AXE 2 — 01. Build Interactions
_Notebook 1/3 — Netflix → `data/warehouse/interactions.parquet`_

## Approche : Virtual Users (mois)
Avec un seul utilisateur réel, ALS PySpark échoue (matrice rang-1 non-définie).  
**Solution** : chaque mois d'activité = un virtual user.

- `user_id` = `year * 100 + month` (ex: 202401 = Janvier 2024)
- ~80 virtual users → ALS peut faire de vraie collaborative filtering

**Output** : `data/warehouse/interactions.parquet`  
Colonnes : `user_id`, `item_id`, `item_title`, `platform`, `play_count`

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Interactions") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"App : {spark.sparkContext.appName}")

# ── Config centrale ────────────────────────────────────────────────────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(_os.path.join(_os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.parquet(os.path.join(WAREHOUSE, name))

Spark version : 3.5.5
App : PySparkShell
Warehouse: /opt/spark/warehouse


26/04/06 14:26:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
# ── 1. NETFLIX VIEWS ──────────────────────────────────────────────────────────
# user_id = year * 100 + month (virtual user mensuel)

netflix_raw = read_table("netflix_views")
print(f"Netflix brut : {netflix_raw.count():,} views")

netflix = netflix_raw.select(
    F.col("show_title").alias("item_title"),
    (F.col("watch_year") * 100 + F.month(F.col("watch_date"))).cast(IntegerType()).alias("user_id")
).filter(
    F.col("item_title").isNotNull() &
    (F.trim(F.col("item_title")) != "") &
    F.col("user_id").isNotNull()
)

print(f"Netflix filtré : {netflix.count():,}")

In [ ]:
# ── 2. AGRÉGATION ─────────────────────────────────────────────────────────────
# Nombre de vues par (virtual_user, item)
# .cache() : interactions_agg est utilisé 2x (stats + filtre bruit)

interactions_agg = netflix.groupBy("user_id", "item_title").agg(
    F.count("*").alias("play_count")
).cache()

n_users    = interactions_agg.select("user_id").distinct().count()
n_items_raw = interactions_agg.select("item_title").distinct().count()
print(f"Virtual users (mois) : {n_users}")
print(f"Items distincts (brut) : {n_items_raw:,}")
print(f"Interactions totales : {interactions_agg.count():,}")

In [6]:
# ── 5. FILTRE BRUIT ───────────────────────────────────────────────────────────
# Exclure les items vus dans < 2 mois distincts (bruit)
# Un item vu plusieurs mois = signal plus fiable

item_month_count = interactions_agg.groupBy("item_title").agg(
    F.countDistinct("user_id").alias("n_months_seen")
).filter(F.col("n_months_seen") >= 2)

interactions_filtered = interactions_agg.join(item_month_count.select("item_title"), on="item_title", how="inner")

n_filtered = interactions_filtered.select("item_title").distinct().count()
print(f"Items après filtre (vus dans >= 2 mois) : {n_filtered:,}")
print(f"Interactions : {interactions_filtered.count():,}")

Items après filtre (vus dans >= 2 mois) : 3,884
Interactions : 15,620


In [ ]:
# ── 4. STRING INDEXER → item_id entier ────────────────────────────────────────
# ALS nécessite des IDs entiers — StringIndexer mappe chaque titre à un index stable.

from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="item_title", outputCol="item_id_float", handleInvalid="keep")
indexer_model = indexer.fit(interactions_filtered)
interactions_indexed = indexer_model.transform(interactions_filtered) \
    .withColumn("item_id", F.col("item_id_float").cast(IntegerType())) \
    .drop("item_id_float")

n_items = interactions_indexed.select("item_id").distinct().count()
print(f"Items indexés : {n_items:,}")
interactions_indexed.orderBy(F.desc("play_count")).show(10, truncate=50)

In [ ]:
# ── 5. ÉCRITURE warehouse/interactions ────────────────────────────────────────

out_df = interactions_indexed.select(
    "user_id",
    "item_id",
    "item_title",
    F.lit("netflix").alias("platform"),
    "play_count"
)

out_path = os.path.join(WAREHOUSE, "interactions")
out_df.write.mode("overwrite").parquet(out_path)

check = spark.read.parquet(out_path)
print(f"Écrit : {out_path}")
print(f"Lignes : {check.count():,}")
check.printSchema()
check.orderBy(F.desc("play_count")).show(10, truncate=50)

spark.stop()
print("Notebook 01 terminé. Lance 02_als_model.ipynb.")